In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {'Baseline_Training', 'CTC_models', 'PhoneticFeatures_Training'}:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import (
    BASELINE_MODELS_DIR,
    CTC_MODELS_DIR,
    DEFAULT_BASELINE_MODEL,
    DEFAULT_CTC_ATTENTION_MODEL,
    DEFAULT_CTC_MODEL,
    DEFAULT_PHONETIC_MODEL_2CLASS,
    DEFAULT_PHONETIC_MODEL_3CLASS,
    EXAMPLE_EMBEDDINGS,
    EXAMPLE_PHONEMES,
    EXAMPLE_SEG_B2,
    EXAMPLE_SEG_B4,
    EXAMPLE_WAV,
    PHONEME_CHOICES_SUMMARY,
    PHONETIC_MODELS_DIR,
    TABLES_DIR,
)


# Attention Phoneme Inference

Ноутбук для инференса `AttentionPhonemeModel`, извлечения attention-матрицы и фонемных границ.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from CTC_tools import (load_selected_embeddings_ctc,
                       CTCPhonemeDataset,
                       ctc_collate_fn,
                       train_ctc,
                       ctc_greedy_decode,
                       CTCModel_lstm,
                       CTCModel_linear,
                       AttentionPhonemeModel)

sys.path.append(r'C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\Main_project')

from evaluation_tools import cer_detailed
# from VizualisationTools import cer_detailed_viz


In [ ]:
example_path = r'C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\Main_project\examples'

embs_file = str(EXAMPLE_EMBEDDINGS)
label_file = str(EXAMPLE_PHONEMES)
wav_file = str(EXAMPLE_WAV)
b2_file = str(EXAMPLE_SEG_B2)

normed_emb = np.load(EXAMPLE_EMBEDDINGS)
normed_emb = torch.tensor(normed_emb, dtype=torch.float32)
normed_emb = normed_emb.T

normed_emb.shape

In [ ]:
# Настройки
CHECKPOINT_PATH = r'" + str(DEFAULT_CTC_ATTENTION_MODEL) + "' # укажите путь к чекпоинту
# CHECKPOINT_PATH = str(CTC_MODELS_DIR / 'LNVMLEZYCY' / 'best_attention_model_monotonic.pth')
EMBEDDING_PATH = str(EXAMPLE_EMBEDDINGS)

# Список фонем должен совпадать с обучением.
# Вставьте сюда тот же phoneme_list, который использовался в training notebook.
phoneme_list_full = ['a0', 'a1', 'a2', 'a4', 'b', "b'", 'c', 'ch', 'ch_', 'd', "d'", 'e0', 'e1', 'e2', 'e4', 'f', "f'", 'g', "g'", 'h', "h'", 'i0', 'i1', 'i2', 'i4', 'j', 'jr', 'jl', 'ji4', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'o1', 'o2', 'o4', 'p', "p'", 'r', "r'", 's', "s'", 'sc', 'sh', 't', "t'", 'u0', 'u1', 'u2', 'u4', 'v', "v'", 'y0', 'y1', 'y2', 'y4', 'z', "z'", 'zh', "zh'", 'C', 'CH', 'H', 'SC']
phoneme_list = ['a0', 'a4', 'b', "b'", 'c', 'ch', 'd', "d'", 'e0', 'f', "f'", 'g', 'h', 'i0','i4', 'j', 'k', "k'", 'l', "l'", 'm', "m'", 'n', "n'", 'o0', 'p', "p'", 'r', "r'", 's', "s'", 'sh', 't', "t'", 'u0', 'v', "v'", 'y0', 'z', "z'", 'zh', 'sil']
phoneme_list = phoneme_list_full + ['sil']

FRAME_MS = 20.0
MAX_STEPS = 120
DECODE_METHOD = "argmax"  # 'center_of_mass' или 'argmax'

INPUT_DIM = None      # если None, возьмется из эмбеддинга
HIDDEN_DIM = 256      # должен совпадать с обученной моделью
NUM_LAYERS = 2
PHONEME_EMB_DIM = 128
DECODER_DIM = 256
ATTENTION_DIM = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
phoneme2idx = {ph: i + 1 for i, ph in enumerate(phoneme_list)}
idx2phoneme = {i + 1: ph for i, ph in enumerate(phoneme_list)}
num_phonemes = len(phoneme_list)

print("num_phonemes:", num_phonemes)


In [ ]:
def load_embedding_sequence(path):
    ext = os.path.splitext(path)[1].lower()

    if ext == ".npy":
        x = np.load(path)
        x = torch.tensor(x, dtype=torch.float32)
    elif ext in [".pt", ".pth"]:
        x = torch.load(path, map_location="cpu")
        if isinstance(x, np.ndarray):
            x = torch.tensor(x, dtype=torch.float32)
        elif not isinstance(x, torch.Tensor):
            raise TypeError("Unsupported tensor object inside .pt/.pth file")
        x = x.float()
    else:
        raise ValueError("Supported formats: .npy, .pt, .pth")

    if x.dim() != 2:
        raise ValueError(f"Expected embedding shape [T, D], got {tuple(x.shape)}")

    return x


def attention_step_times(attention, input_lengths, method="center_of_mass", frame_ms=20.0):
    batch_size, target_len, max_input_len = attention.shape
    frame_positions = torch.arange(max_input_len, device=attention.device, dtype=attention.dtype)
    all_times = []

    for b in range(batch_size):
        T = int(input_lengths[b].item())
        attn = attention[b, :, :T]

        if method == "argmax":
            frame_idx = attn.argmax(dim=-1).float()
        elif method == "center_of_mass":
            weights = attn / attn.sum(dim=-1, keepdim=True).clamp_min(1e-8)
            frame_idx = (weights * frame_positions[:T]).sum(dim=-1)
        else:
            raise ValueError("method must be 'argmax' or 'center_of_mass'")

        times_sec = (frame_idx * frame_ms / 1000.0).detach().cpu().tolist()
        all_times.append(times_sec)

    return all_times


def greedy_decode_attention(model, x_single, idx2phoneme, max_steps=120, start_idx=0, frame_ms=20.0, method="center_of_mass", stop_threshold=0.5):
    model.eval()

    with torch.no_grad():
        x = x_single.unsqueeze(0).to(device)
        input_lengths = torch.tensor([x_single.size(0)], device=device)

        encoder_outputs = model.encode(x)
        encoder_proj = model.encoder_proj(encoder_outputs)

        hidden = encoder_outputs.new_zeros(1, model.decoder_dim)
        cell = encoder_outputs.new_zeros(1, model.decoder_dim)
        context = encoder_outputs.new_zeros(1, encoder_outputs.size(-1))
        current_token = torch.tensor([start_idx], device=device)

        pred_ids = []
        attention_steps = []

        for _ in range(max_steps):
            embedded = model.phoneme_embedding(current_token)
            decoder_input = torch.cat([embedded, context], dim=-1)

            hidden, cell = model.decoder_cell(decoder_input, (hidden, cell))
            context, attn = model._attention(encoder_outputs, encoder_proj, hidden, input_lengths)

            decoder_state = torch.cat([hidden, context], dim=-1)
            logits = model.output_layer(decoder_state)
            stop_logit = model.stop_layer(decoder_state)
            stop_prob = torch.sigmoid(stop_logit).item()
            next_token = logits.argmax(dim=-1)
            token_id = int(next_token.item())

            if token_id == 0:
                break

            pred_ids.append(token_id)
            attention_steps.append(attn.squeeze(0))
            current_token = next_token

            if stop_prob >= stop_threshold:
                break

        if len(pred_ids) == 0:
            return [], [], None

        attention_matrix = torch.stack(attention_steps, dim=0).unsqueeze(0)
        times = attention_step_times(
            attention_matrix,
            input_lengths=input_lengths,
            method=method,
            frame_ms=frame_ms,
        )[0]

        labels = [idx2phoneme.get(idx, "<blank>") for idx in pred_ids]
        pairs = [(float(t), label) for t, label in zip(times, labels)]

        if len(pairs) > 0:
            pairs[0] = (0.0, pairs[0][1])

        return pairs, labels, attention_matrix.squeeze(0)


import matplotlib.pyplot as plt



def plot_attention(attention, labels, frame_ms=20.0, title="Attention alignment"):
    attn = attention.detach().cpu().numpy()
    plt.figure(figsize=(20, 20))
    plt.imshow(
        attn,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="viridis",
    )
    plt.colorbar(label="attention weight")

    plt.xlabel(f"Frame index ({frame_ms} ms each)", fontsize=18)
    plt.ylabel("Phoneme step", fontsize=18)
    plt.title(title, fontsize=20)

    if labels:
        plt.yticks(range(len(labels)), labels, fontsize=16)
    plt.xticks(fontsize=14)

    plt.grid(False)
    plt.tight_layout()
    plt.show()



In [ ]:
#x_single = load_embedding_sequence(EMBEDDING_PATH)
x_single = normed_emb
if INPUT_DIM is None:
    INPUT_DIM = x_single.shape[1]

print("embedding shape:", tuple(x_single.shape))


In [ ]:
attention_model = AttentionPhonemeModel(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    num_phonemes=num_phonemes,
    num_layers=NUM_LAYERS,
    phoneme_emb_dim=PHONEME_EMB_DIM,
    decoder_dim=DECODER_DIM,
    attention_dim=ATTENTION_DIM,
)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
attention_model.load_state_dict(checkpoint["model_state_dict"])
attention_model.to(device)
attention_model.eval()

print("Checkpoint loaded:", CHECKPOINT_PATH)


In [ ]:
result, pred_labels, attention_matrix = greedy_decode_attention(
    model=attention_model,
    x_single=x_single,
    idx2phoneme=idx2phoneme,
    max_steps=MAX_STEPS,
    start_idx=0,
    frame_ms=FRAME_MS,
    method=DECODE_METHOD,
)

result


In [ ]:
if attention_matrix is not None:
    plot_attention(attention_matrix, pred_labels, frame_ms=FRAME_MS)
else:
    print("No phonemes decoded")


In [ ]:
sys.path.append(r'C:\Users\ext-ananeva\phon_clust\gitlab\ssl_phoneme_clusterizer\Main_project')

from VizulalisationTools import plot_waveform_with_vlines_dual_labels
import torchaudio



In [ ]:
waveform, sr = torchaudio.load(wav_file)
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0)

y = waveform.numpy()[0]
duration = y.shape[0] / sr

In [ ]:
target_labelling = []
with open(b2_file, 'r') as f:
    lines = f.readlines()
    print(lines)
    sf = float(lines[1].split('=')[1])
    for i, l in enumerate(lines[7:]):
        start, level, label = l.split(',')
        s = float(start)/(sf*2)
        target_labelling.append((s, label.rstrip('\n')))
target_labelling   

In [ ]:
predicted_labelling = []
predicted_labelling.append(result[0])
max_time = 0
for l in result[1:]:
    print(predicted_labelling[-1][0], max_time)
    
    if l[0] >= max_time:
        
        predicted_labelling.append(l)
        max_time = l[0]
    else:
        
        break
        

In [ ]:
len(predicted_labelling), len(target_labelling)

In [ ]:

plot_waveform_with_vlines_dual_labels(y, sr, target_labelling, predicted_labelling)

In [ ]:
diffs = []
if len(target_labelling)>len(predicted_labelling):
    enumerate_arr = target_labelling
else:
    enumerate_arr = predicted_labelling
    

for i, l in enumerate(enumerate_arr):
    dif = abs(target_labelling[i][0] - predicted_labelling[i][0])
    diffs.append(dif)
    
print(diffs)

In [ ]:
sum(diffs)/len(diffs)

In [ ]:
diff = []
res = []
for i, j in zip(target_labelling, predicted_labelling):
    diff.append(abs(i[0] - j[0]))
    res.append(1 if i[1]==j[1] else 0)

In [ ]:
np.mean(diff) # 0.017649554098752695

In [ ]:
predicted_labelling

In [ ]:
np.mean(res)